# Field sweeps on the Teslatron

Longitudinal resistance of a graphene Hall bar against magnetic field, at four
temperatures and two sample orientations. At each temperature the rotator moves
to an angle, the field sweeps from -6 T to 6 T, and a row is written for every
field. The run ends at zero field, base temperature and zero angle.

### Instruments

| Instrument | Role |
| --- | --- |
| Mercury iTC | VTI and probe temperature, needle valve |
| Mercury iPS | magnet |
| Rotation probe | sample angle |
| SR830 | reads the sample |

### Wiring

`I+ 24, I- 12, Vxx+ 8, Vxx- 9`

Excitation is 100 nA, from the lock-in oscillator through a 10 megohm series
resistor. The measurement is four-terminal, so the voltage leads carry no
current and the contact resistance stays out of the result.

### What the data shows

Only the field component perpendicular to the flake acts on a two-dimensional
sample. At 0 degrees the field is perpendicular, and the 2 K trace carries
Shubnikov-de Haas oscillations, periodic in 1/B, which wash out by 20 K once
thermal smearing exceeds the Landau level spacing. At 90 degrees the field lies
in the plane and the sweep is nearly flat.

In [ ]:
### Library Import ###
import csv
import datetime
import os
import pathlib

import matplotlib.pyplot as plt

from labdrivers.funky_rotator import Rotator
from labdrivers.oxford import MercuryIpsTeslatron, MercuryItc
from labdrivers.srs import Sr830

## Settings

Everything that changes between runs. The addresses, the sweep, the lock-in, and
where the data goes.

In [ ]:
### Settings ###
ips_address = "192.168.0.10"
itc_address = "192.168.0.11"
lockin_gpib = 8
rotator_axis = 1

temperatures = [2.0, 5.0, 10.0, 20.0]
angles = [0.0, 90.0]  # 0 is field perpendicular to the flake
field_max = 6.0
field_points = 121
ramp_rate = 0.3  # T/min, within what this magnet is rated for
settle = 2.0  # extra seconds at each field, for eddy currents in the probe
magnet_temp_limit = 3.95  # the magnet is rated to 4.2 K

amplitude = 1.0
resistor = 10e6
frequency = 17.777  # away from every line harmonic
time_constant = 0.3
sensitivity = 1e-3

tolerance = 0.02  # a fraction of the target, so 40 mK at 2 K
hold = 60.0
timeout = 3600.0

# VTI pressure by probe temperature, in mbar. A different VTI wants its own.
vti_pressures = [(2.0, 5.0), (7.0, 6.8), (float("inf"), 20.0)]

wiring = "I+ 24, I- 12, Vxx+ 8, Vxx- 9"
data_directory = pathlib.Path.home() / "measurements"

## Temperature

The VTI needs its needle valve set for the range it is working in, so the
pressure changes with the setpoint. `wait_for_temperature` returns once the
probe has stayed inside the tolerance band for a full minute, since a cryostat
passes through its setpoint on the way.

In [ ]:
### Auxiliary Functions ###
def add_point(line, axis, x, y):
    """Add one point to a live plot line and rescale."""
    xs, ys = line.get_data()
    line.set_data([*xs, x], [*ys, y])
    axis.relim()
    axis.autoscale_view()
    plt.pause(0.01)


def set_temperature(itc, target):
    """Set the VTI pressure and both setpoints, then wait for the probe."""
    for below, mbar in vti_pressures:
        if target < below:
            itc.pressure_setpoint = mbar
            break
    itc.setpoint("vti", target)
    itc.setpoint("probe", target)
    itc.wait_for_temperature(
        "probe", target, tolerance=tolerance, hold=hold, timeout=timeout
    )

## The measurement

Temperature is the outer loop, angle the middle, field the inner. The magnet
returns to zero before every rotation, because 6 T puts real torque on the
probe.

Rows are written as they arrive rather than collected and saved at the end, so a
run interrupted after two hours still has two hours on disk. The shutdowns sit
in a `finally`: closing a connection is not the same as ramping a magnet down,
and a `with` block only does the first.

In [ ]:
### Measurement ###
current = amplitude / resistor
started = datetime.datetime.now()
data_directory.mkdir(exist_ok=True)
path = data_directory / f"teslatron_{started:%Y-%m-%d_%H%M}.csv"

# "x" rather than "a", so a rerun cannot quietly append to an old run
file = open(path, "x", newline="")
file.write(f"# {started:%Y-%m-%d %H:%M}, wiring {wiring}, {current:.2e} A\n")
file.write(f"# {temperatures} K, {angles} deg, +/-{field_max} T\n")
writer = csv.writer(file)
writer.writerow(
    ["temperature_set_K", "angle_deg", "field_T", "probe_K", "vti_K", "magnet_K",
     "x_V", "y_V", "resistance_ohm", "overloaded"]
)

plt.ion()
figure, axes = plt.subplots(1, len(temperatures), figsize=(14, 4), sharey=True)
for axis, temperature in zip(axes, temperatures):
    axis.set_title(f"{temperature:g} K")
    axis.set_xlabel("field (T)")
axes[0].set_ylabel("Rxx (ohm)")

with MercuryIpsTeslatron(ip_address=ips_address) as supply, MercuryItc(
    ip_address=itc_address
) as itc, Sr830(gpib_address=lockin_gpib) as lockin, Rotator(
    axis=rotator_axis
) as rotator:
    # a wrong address fails here rather than three steps in
    print(lockin.identify())
    print(f"probe {itc.temperature('probe'):.3f} K, magnet {supply.magnet_temperature:.2f} K")

    lockin.reference_source = "internal"
    lockin.frequency = frequency
    lockin.amplitude = amplitude
    lockin.input_configuration = "a-b"
    lockin.time_constant = time_constant
    lockin.sensitivity = sensitivity

    supply.magnet.field_ramp_rate = ramp_rate
    supply.magnet.switch_heater = True
    supply.wait_for_switch_heater(True)

    try:
        for axis, temperature in zip(axes, temperatures):
            set_temperature(itc, temperature)
            for angle in angles:
                supply.magnet.ramp_to_field(0.0)
                rotator.move_to(angle)
                if supply.magnet_temperature > magnet_temp_limit:
                    raise RuntimeError(
                        f"The magnet is at {supply.magnet_temperature} K and the "
                        f"limit is {magnet_temp_limit} K."
                    )

                (line,) = axis.plot([], [], label=f"{angle:g} deg")
                axis.legend()

                for field in supply.magnet.sweep_field(
                    -field_max, field_max, points=field_points, settle=settle
                ):
                    x, y = lockin.measure()
                    resistance = (x**2 + y**2) ** 0.5 / current
                    writer.writerow(
                        [temperature, rotator.angle, field, itc.temperature("probe"),
                         itc.temperature("vti"), supply.magnet_temperature, x, y,
                         resistance, int(lockin.overloaded())]
                    )
                    file.flush()
                    os.fsync(file.fileno())
                    add_point(line, axis, field, resistance)
    finally:
        supply.safe_shutdown()
        rotator.move_to(0.0)
        itc.heater_enabled("probe", False)
        itc.heater_enabled("vti", False)
        file.close()
        figure.savefig(path.with_suffix(".png"))

print("Run finished")

## Output

One CSV and one PNG in the data directory, both named for the time the run
started. Every row carries what was set and what was read back, including all
three thermometers and the lock-in overload flag, so a suspect point can be
checked afterwards.

![Rxx against field at four temperatures](teslatron_field_sweeps.png)

The plot above uses made-up data, shaped like a real run.

The file starts with two comment lines, so pandas needs `comment="#"`:

```python
import pandas as pd

data = pd.read_csv(path, comment="#")
perpendicular = data[data.angle_deg == 0]
```